# Render the failed Phase-5 blades (the self-intersecting geometry)

**Fresh Colab session — reads your Drive, re-runs no phase.** It loads the design *numbers* Phase 4
saved (`checkpoint.npz`), finds which designs Phase 5 flagged as `SUSPECT`
(`verification.json`), rebuilds each blade's 3D CAD solid with CadQuery, and renders it so you can
**see** where it crosses through itself. CadQuery only (no gmsh/SU2) — the CAD solid is what
gmsh choked on, so we can still build and draw it.

## 1. Repo + deps + mount Drive

In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    REPO = Path("/content/fan-optimization")
    if not REPO.exists():
        subprocess.run(["git","clone","-b","main","https://github.com/clingergab/fan-optimization.git",str(REPO)], check=True)
    else:
        subprocess.run(["git","-C",str(REPO),"pull","origin","main"], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","cadquery","matplotlib"], check=True)
    from google.colab import drive; drive.mount("/content/drive")
else:
    REPO = Path.cwd()
    while REPO != REPO.parent and not (REPO/"pyproject.toml").exists(): REPO = REPO.parent
if str(REPO/"src") not in sys.path: sys.path.insert(0, str(REPO/"src"))
print("repo:", REPO, "| colab:", IN_COLAB)

## 2. Where the saved data lives

`checkpoint.npz` holds every design's 35 numbers (`x`); `verification.json` says which failed.
Point these at your Drive folders.

In [ ]:
import numpy as np, json
CAMPAIGN = Path("/content/drive/MyDrive/fanopt/phase4_bo") if IN_COLAB else REPO/"data"/"phase4_bo_nb"
VERIFY   = Path("/content/drive/MyDrive/fanopt/phase5_verify") if IN_COLAB else REPO/"data"/"phase5_verify_nb"

x = np.load(CAMPAIGN/"checkpoint.npz")["x"]                    # (n_designs, 35) design vectors
ranking = json.loads((VERIFY/"verification.json").read_text())["ranking"]
suspects = ranking.get("suspect_designs", [])                  # e.g. ['b8_i154','b8_i168','b8_i197']
print(f"{x.shape[0]} designs on file | suspects to render: {suspects}")

## 3. Rebuild + render one blade

Design numbers → `decode` → `make_vunit_blade` (the CAD solid) → `tessellate` (turn the solid into
triangles) → draw the triangles in 3D. The design name `b8_i154` means campaign index 154, i.e.
row `x[154]`.

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.pyplot as plt
from fanopt.bo.codec import decode
from fanopt.geometry.generator import BladeDesignParams
from fanopt.geometry.fields import Layer2Params
from fanopt.geometry.primitives import Layer3Primitive
from fanopt.bo.inertia import NEUTRAL_LAYER4
from fanopt.geometry.assembly_cad import make_vunit_blade

def blade_triangles(vec, tol=0.001):
    params = BladeDesignParams(layer1=decode(vec), layer2=Layer2Params.all_inactive(),
                               layer3=Layer3Primitive.absent(), layer4=NEUTRAL_LAYER4)
    shape = make_vunit_blade(params).val()          # CadQuery solid
    verts, tris = shape.tessellate(tol)             # -> (points, triangle indices)
    V = np.array([[p.x, p.y, p.z] for p in verts])
    return V, [V[list(t)] for t in tris]

def render(ax, name, vec):
    V, polys = blade_triangles(vec)
    ax.add_collection3d(Poly3DCollection(polys, alpha=0.55, facecolor="crimson",
                                         edgecolor="k", linewidth=0.05))
    for lo, hi, setl in zip(V.min(0), V.max(0), (ax.set_xlim, ax.set_ylim, ax.set_zlim)):
        setl(lo, hi)
    ax.set_box_aspect((1,1,1)); ax.set_title(name); ax.view_init(elev=25, azim=-70)

## 4. Render every failed design

Rotate each one (drag). The self-intersecting ones show surfaces passing through the body —
that's the `overlapping facets` gmsh rejected. (If a design was flagged suspect only for negative
airflow, it'll look like a *normal* blade — its geometry was fine, its physics wasn't.)

In [ ]:
names = suspects or []
if not names:
    print("No suspects in verification.json — nothing to render.")
else:
    fig = plt.figure(figsize=(6*len(names), 6))
    for i, name in enumerate(names):
        idx = int(name.split("_i")[-1])
        ax = fig.add_subplot(1, len(names), i+1, projection="3d")
        try:
            render(ax, name, x[idx])
        except Exception as e:
            ax.set_title(f"{name}: CAD build failed\n{type(e).__name__}")
    plt.tight_layout(); plt.show()

## 5. Takeaway

These are your **actual** failed designs, not a toy. Compare to a passing design (swap a valid
name into `names` above) — the valid ones are clean sheets, the failures fold through themselves.
This is exactly the geometry the **V1.5 validity filter** (`docs/V2_backlog.md`) would reject
*before* wasting a 3D run on it.